In [1]:
import os

print("Looking for files in /kaggle/input/...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Looking for files in /kaggle/input/...
/kaggle/input/awscc-recruitment/fake news/sample_submission.csv
/kaggle/input/awscc-recruitment/fake news/WELFake_Dataset.csv
/kaggle/input/awscc-recruitment/fake news/test.csv


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv(
    '/kaggle/input/awscc-recruitment/fake news/WELFake_Dataset.csv',
    encoding='latin-1',usecols=[0,1,2,3]
)
test = pd.read_csv('/kaggle/input/awscc-recruitment/fake news/test.csv', encoding='latin-1')
sample_sub = pd.read_csv('/kaggle/input/awscc-recruitment/fake news/sample_submission.csv')

print(train.shape)
print(train.columns.tolist())
print(train.head())
print(train['label'].dtype)
print(train['label'].unique()[:20])
train['label'] = pd.to_numeric(train['label'], errors='coerce')
print(train['label'].value_counts())
print(f"{train['label'].isna().sum()}")
train = train.dropna(subset=['label'])
train['label'] = train['label'].astype(int)
print(f"{train.shape}")
print(train['label'].value_counts())
print(train.isnull().sum())
print(test.shape)
print(test.head())
print(sample_sub.head())

(78207, 4)
['id', 'title', 'text', 'label']
    id                                              title  \
0  150  On âBrexit,â Itâs Divorce First, Trade T...   
1  151  Ingraham: The âTrump-iestâ Parts of Obamac...   
2  152  Barack Obamaâs Final Arms-Export Totals Doub...   
3  153        The Glorious New PC NFL Starts Next Monday!   
4  154  Indian court acquits dentist couple of killing...   

                                                text label  
0  BRUSSELS  â   Britain must agree to pay its ...     0  
1  .@IngrahamAngle: The â â parts of Obamacar...     0  
2  21st Century Wire says 21WIRE has always point...     1  
3  Afraid of "dangerous" Trump presidency, protes...     1  
4  NEW DELHI (Reuters) - An Indian court on Thurs...     0  
object
['0' '1' nan
 ' Duncan explained â\x80\x9cBaiame came from a place that we call the Morning Star within the Mirrabooka. Mira means stars and booka means river. That is the Milky Way that flows across the North Star. 

/tmp/ipykernel_17/3775265364.py:4: DtypeWarning: Columns (0,3) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(


In [3]:
train = train[train['label'].isin([0, 1])].copy()
train['label'] = train['label'].astype(int)
print(f"{train.shape}")
print(f"{train['label'].value_counts()}")
train['title'] = train['title'].fillna('')
train['text'] = train['text'].fillna('')
test['title'] = test['title'].fillna('')
test['text'] = test['text'].fillna('')
train['content'] = train['title'] + ' ' + train['text']
test['content'] = test['title'] + ' ' + test['text']
print(train['content'].str.len().describe())
print(train['content'].iloc[0][:300])
print(f"{test.shape}")

(71868, 4)
label
1    36955
0    34913
Name: count, dtype: int64
count    71868.000000
mean      3296.647562
std       3090.259527
min          2.000000
25%       1465.000000
50%       2513.000000
75%       4163.000000
max      32712.000000
Name: content, dtype: float64
On âBrexit,â Itâs Divorce First, Trade Talks Later, E.U. Tells U.K. - The New York Times BRUSSELS  â   Britain must agree to pay its bills and to protect millions of Europeans living in Britain before reaching a new trading relationship with the European Union, Donald Tusk, the president of 
(150, 4)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np
tfidf = TfidfVectorizer(
    max_features=80000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode'
)

X_train = tfidf.fit_transform(train['content'])
X_test = tfidf.transform(test['content'])
y_train = train['label'].values
print(f"{X_train.shape}")
print(f"{X_test.shape}")
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1,
    random_state=42
)
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"accuracy: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})")
print(f"Per-fold scores: {[f'{s:.5f}' for s in cv_scores]}")
lr.fit(X_train, y_train)
test_preds = lr.predict(X_test)
print(f"{(test_preds == 0).sum()}")
print(f"{(test_preds == 1).sum()}")

(71868, 80000)
(150, 80000)
accuracy: 0.96729 (+/- 0.00160)
Per-fold scores: ['0.96758', '0.96828', '0.96563', '0.96535', '0.96960']
66
84


In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
f1_scores_lr = cross_val_score(lr, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
print(f"Baseline LR F1: {f1_scores_lr.mean():.5f} (+/- {f1_scores_lr.std():.5f})")
for C in [0.5, 1.0, 2.0, 5.0, 10.0]:
    lr_temp = LogisticRegression(C=C, max_iter=1000, solver='lbfgs', n_jobs=-1, random_state=42)
    scores = cross_val_score(lr_temp, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LR C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")
svc = LinearSVC(C=0.5, max_iter=2000, class_weight='balanced', random_state=42)
f1_svc = cross_val_score(svc, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
print(f"\nLinearSVC F1: {f1_svc.mean():.5f} (+/- {f1_svc.std():.5f})")
tfidf_large = TfidfVectorizer(
    max_features=150000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode'
)
X_train_large = tfidf_large.fit_transform(train['content'])
X_test_large = tfidf_large.transform(test['content'])
print(f"\nLarger TF-IDF shape: {X_train_large.shape}")
for C in [0.5, 1.0, 2.0, 5.0]:
    lr_temp = LogisticRegression(C=C, max_iter=1000, solver='lbfgs', n_jobs=-1, random_state=42)
    scores = cross_val_score(lr_temp, X_train_large, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LR (large) C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")
for C in [1.0, 2.0, 5.0]:
    lr_temp = LogisticRegression(C=C, max_iter=1000, solver='lbfgs', n_jobs=-1, 
                                  random_state=42, class_weight='balanced')
    scores = cross_val_score(lr_temp, X_train_large, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LR (large, balanced) C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")

Baseline LR F1: 0.96840 (+/- 0.00151)
LR C=0.5: F1=0.96358 (+/- 0.00205)
LR C=1.0: F1=0.96840 (+/- 0.00151)
LR C=2.0: F1=0.96857 (+/- 0.00135)
LR C=5.0: F1=0.97685 (+/- 0.00168)
LR C=10.0: F1=0.97842 (+/- 0.00236)

LinearSVC F1: 0.98057 (+/- 0.00121)

Larger TF-IDF shape: (71868, 150000)
LR (large) C=0.5: F1=0.96278 (+/- 0.00158)
LR (large) C=1.0: F1=0.96789 (+/- 0.00108)
LR (large) C=2.0: F1=0.97118 (+/- 0.00203)
LR (large) C=5.0: F1=0.97514 (+/- 0.00276)
LR (large, balanced) C=1.0: F1=0.96535 (+/- 0.00405)
LR (large, balanced) C=2.0: F1=0.97238 (+/- 0.00126)
LR (large, balanced) C=5.0: F1=0.97764 (+/- 0.00089)


In [6]:
for C in [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]:
    svc = LinearSVC(C=C, max_iter=3000, random_state=42)
    scores = cross_val_score(svc, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LinearSVC C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")
for C in [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    svc = LinearSVC(C=C, max_iter=3000, class_weight='balanced', random_state=42)
    scores = cross_val_score(svc, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LinearSVC balanced C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")
for C in [10, 15, 20, 30, 50]:
    lr_temp = LogisticRegression(C=C, max_iter=2000, solver='lbfgs', n_jobs=-1, random_state=42)
    scores = cross_val_score(lr_temp, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"LR C={C}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")
from sklearn.linear_model import SGDClassifier
for alpha in [1e-4, 5e-5, 1e-5, 5e-6, 1e-6]:
    sgd = SGDClassifier(loss='modified_huber', alpha=alpha, max_iter=2000, 
                        random_state=42, n_jobs=-1)
    scores = cross_val_score(sgd, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"SGD alpha={alpha}: F1={scores.mean():.5f} (+/- {scores.std():.5f})")

LinearSVC C=0.1: F1=0.97257 (+/- 0.00107)
LinearSVC C=0.3: F1=0.97896 (+/- 0.00077)
LinearSVC C=0.5: F1=0.98047 (+/- 0.00109)
LinearSVC C=0.7: F1=0.98121 (+/- 0.00117)
LinearSVC C=1.0: F1=0.98174 (+/- 0.00101)
LinearSVC C=1.5: F1=0.98193 (+/- 0.00102)
LinearSVC C=2.0: F1=0.98201 (+/- 0.00094)
LinearSVC C=3.0: F1=0.98181 (+/- 0.00084)
LinearSVC balanced C=0.1: F1=0.97275 (+/- 0.00096)
LinearSVC balanced C=0.3: F1=0.97919 (+/- 0.00082)
LinearSVC balanced C=0.5: F1=0.98057 (+/- 0.00121)
LinearSVC balanced C=0.7: F1=0.98132 (+/- 0.00113)
LinearSVC balanced C=1.0: F1=0.98187 (+/- 0.00104)
LinearSVC balanced C=1.5: F1=0.98200 (+/- 0.00099)
LinearSVC balanced C=2.0: F1=0.98202 (+/- 0.00089)
LR C=10: F1=0.97842 (+/- 0.00236)
LR C=15: F1=0.97878 (+/- 0.00252)
LR C=20: F1=0.97890 (+/- 0.00262)
LR C=30: F1=0.97829 (+/- 0.00374)
LR C=50: F1=0.97382 (+/- 0.00131)
SGD alpha=0.0001: F1=0.97646 (+/- 0.00062)
SGD alpha=5e-05: F1=0.97975 (+/- 0.00099)
SGD alpha=1e-05: F1=0.98185 (+/- 0.00093)
SGD alpha=

In [7]:
from sklearn.svm import LinearSVC
model = LinearSVC(C=2.0, max_iter=3000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
X_test_text = test['title'].fillna('')
X_test = tfidf.transform(X_test_text)
predictions = model.predict(X_test)
submission = pd.DataFrame({
    'id': test['id'],
    'label': predictions
})
submission.to_csv('submission.csv', index=False)

In [8]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
print(f"CV F1: {scores.mean():.4f} ± {scores.std():.4f}")
print(test.columns.tolist())
print(test[['title']].head(10))

CV F1: 0.9820 ± 0.0009
['id', 'title', 'text', 'content']
                                               title
0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1                                                   
2  UNBELIEVABLE! OBAMAÃ¢ÂÂS ATTORNEY GENERAL SA...
3  Bobby Jindal, raised Hindu, uses story of Chri...
4  SATAN 2: Russia unvelis an image of its terrif...
5  About Time! Christian Group Sues Amazon and SP...
6  DR BEN CARSON TARGETED BY THE IRS: Ã¢ÂÂI nev...
7  HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...
8  Sports Bar Owner Bans NFL GamesÃ¢ÂÂ¦Will Show...
9  Latest Pipeline Leak Underscores Dangers Of Da...


In [9]:
train['content'] = train['title'].fillna('') + ' ' + train['text'].fillna('')
test['content'] = test['title'].fillna('') + ' ' + test['text'].fillna('')
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
X_train = tfidf.fit_transform(train['content'])
y_train = train['label']
X_test = tfidf.transform(test['content'])
model = LinearSVC(C=1.0, max_iter=3000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(pd.Series(predictions).value_counts())

1    85
0    65
Name: count, dtype: int64


In [10]:
test_predictions = model.predict(X_test)
submission = pd.DataFrame({
    'id': test['id'],
    'label': test_predictions
})
submission.to_csv('submission.csv', index=False)